In [1]:
import glob
import torch
import numpy as np
import scipy as sp
import scipy.io
import scipy.signal
import time
import pandas as pd
import math
from utils.data_utils import *
from utils.viz_utils import *

from torch import nn
import torch.optim as optim
from scipy.io import loadmat
import matplotlib.pyplot as plt
from matplotlib import figure
from scipy.signal import savgol_filter
from scipy.signal import butter, lfilter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
torch.manual_seed(42)
np.random.seed(42)

fs = 125
minBPM = 40
maxBPM = 240
window_length = 8 * fs
window_shift = 2 * fs  #Overlap = window_length - window_shift

# Retrieve dataset files
data_dir = "datasets/troika/training_data"
data_fls, ref_fls = LoadTroikaDataset(data_dir)
errs, confs = [], []

device = (
    "cuda"
    if torch.cuda.is_available()
    # else "mps"
    # if torch.backends.mps.is_available()
    else "cpu"
)
# device = "cpu"
print(f"Using {device} device")

batch_size = 25

# Create the dataset
dataset = TroikaDataset(data_fls, ref_fls, window_length, window_shift, fs)

# Split the dataset into training and testing
train_size = 0.8
num_train = int(len(dataset) * train_size)
num_test = len(dataset) - num_train
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [num_train, num_test])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class motionPPGNet(nn.Module):
    def __init__(self, n_timesteps, n_features, n_outputs):
        super(motionPPGNet, self).__init__()
        # Conv1D blocks
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=32, kernel_size=40)
        self.bn1 = nn.BatchNorm1d(num_features=32)
        self.pool1 = nn.MaxPool1d(kernel_size=4)
        self.dropout1 = nn.Dropout(p=0.1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=32, kernel_size=40)
        self.bn2 = nn.BatchNorm1d(num_features=32)
        self.pool2 = nn.MaxPool1d(kernel_size=4)
        self.dropout2 = nn.Dropout(p=0.1)
        
        # Calculate the sequence length after conv and pooling
        l1 = n_timesteps - 40 + 1  # Conv1 output length
        l2 = l1 // 4              # Pool1 output length
        l3 = l2 - 40 + 1          # Conv2 output length
        self.seq_len_after_conv = l3 // 4  # Pool2 output length
        print(f"Sequence length after convolutions: {self.seq_len_after_conv}")
        
        # LSTM layers
        self.lstm1 = nn.LSTM(input_size=32, hidden_size=128, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=128, hidden_size=128, batch_first=True)
        
        # Final dense layer for output
        self.dense = nn.Linear(in_features=128, out_features=1)
        
        # Store activations
        self.activations = {}
        
        # Initialize weights using Xavier/Glorot (similar to TensorFlow default)
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
      
    def forward(self, x):
        # x shape: (batch_size, n_features, n_timesteps)
        
        # Convolutional layers
        x = self.conv1(x)
        x = torch.relu(self.bn1(x))
        x = self.pool1(x)
        x = self.dropout1(x)
        
        x = self.conv2(x)
        x = torch.relu(self.bn2(x))
        x = self.pool2(x)
        x = self.dropout2(x)
        
        # Reshape for LSTM: (batch_size, seq_len, features)
        x = x.permute(0, 2, 1)
        
        # LSTM layers
        x, (h1, c1) = self.lstm1(x)
        x, (h2, c2) = self.lstm2(x)
        
        # Use the final hidden state
        x = h2.squeeze(0)
        
        # Final dense layer
        x = self.dense(x)
        
        return x.squeeze(-1)  # Remove last dim if it's 1



verbose, epochs = 1, 100
n_timesteps, n_features, n_outputs = len(train_dataset[0][0][0]), 1, 1
learning_rate = 1e-3
# epochs = 5

# Initialize the model
model = motionPPGNet(n_timesteps, n_features, n_outputs)
model.to(device)

# Loss and optimizer
criterion = nn.MSELoss()
learning_rate = 1e-3  # Reduce by 10x
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
# optimizer = optim.Adam(model.parameters(), lr=0.001)
# optimizer = optim.RMSprop(model.parameters(), lr=learning_rate)

print(n_timesteps)


# Training loop
for epoch in range(1, epochs+1):
    model.train()
    epoch_train_loss = 0
    train_batches = 0
    for ppg, acc, targets in train_loader:
        # print(ppg)
        # print(ppg.shape)
        inputs, targets = ppg.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        epoch_train_loss += loss.item()
        train_batches += 1
    # break
    avg_train_loss = epoch_train_loss/train_batches

    if epoch % 10 == 0:  # Validation every 10 epochs
        model.eval()
        val_abs_error = 0
        val_batches = 0
        val_samples = 0
        with torch.no_grad():
            total_loss = 0
            for ppg, _, targets in test_loader:
                inputs, targets = ppg.to(device), targets.to(device)
                outputs = model(inputs)
                # print(f"Outputs: {outputs},\n Targets: {targets}")
                loss = criterion(outputs, targets)
                abs_error = torch.abs(outputs - targets).sum().item()
                val_abs_error += abs_error
                val_batches +=1
                val_samples += targets.size(0)
                total_loss += loss.item()
            print(f'Epoch {epoch}:')
            print(f"Training Loss: {avg_train_loss:.2f}")
            print(f"Test&Val Loss: {total_loss / len(test_loader):.2f}")
            print(f"Val MAE: {val_abs_error / val_samples:.2f}")
            # Calculate training/validation loss ratio to help identify overfitting
            loss_ratio = avg_train_loss / (total_loss / len(test_loader)) if (total_loss / len(test_loader)) > 0 else 1.0
            print(f'  Train/Val Loss Ratio: {loss_ratio:.4f} (< 0.8 may indicate underfitting, > 1.2 may indicate overfitting)')



Using cpu device
Sequence length after convolutions: 50


/Users/tmagcaya/miniconda3/envs/torch/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1000
Epoch 10:
Training Loss: 3277.51
Test&Val Loss: 3240.92
Val MAE: 51.91
  Train/Val Loss Ratio: 1.0113 (< 0.8 may indicate underfitting, > 1.2 may indicate overfitting)
Epoch 20:
Training Loss: 417.97
Test&Val Loss: 405.06
Val MAE: 16.82
  Train/Val Loss Ratio: 1.0319 (< 0.8 may indicate underfitting, > 1.2 may indicate overfitting)
Epoch 30:
Training Loss: 207.31
Test&Val Loss: 228.29
Val MAE: 11.53
  Train/Val Loss Ratio: 0.9081 (< 0.8 may indicate underfitting, > 1.2 may indicate overfitting)
Epoch 40:
Training Loss: 121.46
Test&Val Loss: 168.35
Val MAE: 9.29
  Train/Val Loss Ratio: 0.7215 (< 0.8 may indicate underfitting, > 1.2 may indicate overfitting)
Epoch 50:
Training Loss: 86.09
Test&Val Loss: 98.04
Val MAE: 7.00
  Train/Val Loss Ratio: 0.8782 (< 0.8 may indicate underfitting, > 1.2 may indicate overfitting)
Epoch 60:
Training Loss: 53.12
Test&Val Loss: 110.28
Val MAE: 6.91
  Train/Val Loss Ratio: 0.4817 (< 0.8 may indicate underfitting, > 1.2 may indicate overfitting)
Epo

In [2]:
import torch

mps_device = torch.device("mps")

x = 0.1

cpu_tensor = torch.exp(torch.tensor(x))
mps_tensor = torch.exp(torch.tensor(x, device=mps_device))

print(cpu_tensor - cpu_tensor) # prints 0
print(mps_tensor - mps_tensor) # prints 0
print(cpu_tensor - mps_tensor) # prints 1.1921e-07
print(cpu_tensor - mps_tensor.cpu()) # prints 1.1921e-07
print(cpu_tensor.to(mps_device) - mps_tensor) # prints 1.1921e-07

tensor(0.)
tensor(0., device='mps:0')
tensor(0., device='mps:0')
tensor(0.)
tensor(0., device='mps:0')
